# Week 5 — Decision Tree and Random Forest comparison
Same intrinsic predictors, cleaning, training-only preprocessing and target bounds as Weeks 1–4. December–March trains; April selects by maximum R². May was already inspected for the original baseline and is not a pristine holdout.

## 1. Protocol and implementation
Three candidates per tree family keep computation modest. The first is a near-default starting point with a 60-tree forest; alternatives regularize depth/leaves and vary feature subsampling. Random seed 42; forest uses two workers. See reusable code below.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image
from src import experiments
OUT = ROOT / 'outputs'
import inspect
display(pd.DataFrame([(name, params) for name, grid in experiments.GRIDS.items() for params in grid], columns=['model', 'parameters']))
print(inspect.getsource(experiments.fit_candidates))

## 2. Train, tune on April, freeze, then evaluate May
All imputers, category grouping, scaling and model fitting see training only. Highest April R² selects each family and the overall winner; ties retain grid order. No April refit before May. Re-running reproduces this fixed experiment, not a new untouched test.

In [ ]:
comparison, may = experiments.run_week5(ROOT)
display(comparison)
print('Frozen winner May:', may)

## 3. Tuning evidence and baseline reproduction
Compare train/April gaps as well as R² and percentage/dollar errors. The runner checks recomputed baseline validation against historical tolerances without overwriting baseline artifacts.

In [ ]:
display(pd.read_csv(OUT / 'week05_tuning.csv'))
print((OUT / 'week05_selection.json').read_text())

## 4. Model comparisons and error diagnostics
Bar charts compare train and April. Scatter and signed percentage error charts describe only the selected frozen model on bounded May. Importance is training impurity reduction, biased toward flexible/high-cardinality variables, not causal evidence.

In [ ]:
for path in sorted((OUT / 'figures').glob('week05_*.png')):
    print(path.name)
    display(Image(filename=str(path)))

## 5. Findings and limitations
Open `docs/WEEK05_FINDINGS.md` for verified values and the meeting guide. Bounded metrics exclude unknown-at-prediction target tails; all-cleaned May sensitivity is mandatory. One April selection month gives no confidence interval or proof of future stability. All proprietary data and fitted artifacts stay local.